# 🛒 E-Commerce Sales Analysis
### Olist Dataset — End-to-End Data Analysis Project

This notebook analyses the **Olist Brazilian e-commerce dataset**.  
It covers the full pipeline: data loading → cleaning → analysis → visualisation → business insights.

---

## 📂 Section 1 — Data Loading

In [ ]:
import pandas as pd

# Load all five source tables from CSV files
orders       = pd.read_csv("/content/olist_orders_dataset.csv")
order_items  = pd.read_csv("/content/olist_order_items_dataset.csv")
payments     = pd.read_csv("/content/olist_order_payments_dataset.csv")
customers    = pd.read_csv("/content/olist_customers_dataset.csv")
products     = pd.read_csv("/content/olist_products_dataset.csv")

In [ ]:
# Quick look at the orders table — structure and data types
orders.head()

In [ ]:
orders.info()

## 🧹 Section 2 — Data Cleaning

In [ ]:
# Convert purchase timestamp to datetime format for time-based analysis
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])

In [ ]:
# Check for missing values in the orders table
orders.isnull().sum()

In [ ]:
# See how many orders fall under each status
orders["order_status"].value_counts()

In [ ]:
# Keep only delivered orders — these have complete fulfilment data
orders_delivered = orders[orders["order_status"] == "delivered"]

In [ ]:
# Check missing values in the delivered-only subset
orders_delivered.isnull().sum()

In [ ]:
# Drop rows that still have missing values after the status filter
orders_clean = orders_delivered.dropna()

### Merging all tables into a single master dataframe

In [ ]:
# Merge customers onto orders
df = orders_clean.merge(customers,   on="customer_id",  how="left")

# Merge payment information
df = df.merge(payments,   on="order_id",  how="left")

# Merge order line items
df = df.merge(order_items, on="order_id",  how="left")

# Merge product details
df = df.merge(products,    on="product_id", how="left")

print("Merged dataframe shape:", df.shape)

In [ ]:
# Check for remaining missing values across all columns
df.isnull().sum()

In [ ]:
# Drop columns that are not needed for this analysis
df = df.drop(columns=[
    "product_name_lenght",       # Not used in analysis
    "product_description_lenght", # Not used in analysis
    "product_photos_qty"          # Not used in analysis
])

In [ ]:
# Fill missing product category names with 'Unknown' to avoid losing rows
df["product_category_name"].fillna("Unknown", inplace=True)

In [ ]:
# Drop physical dimension columns — not relevant for revenue analysis
df = df.drop(columns=[
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
])

In [ ]:
# Final check — confirm no missing values remain
df.isnull().sum()

In [ ]:
# Save the cleaned dataframe to CSV for later use
df.to_csv("cleaned_ecommerce_data.csv", index=False)
print("✅ Cleaned data saved successfully.")

## 📊 Section 3 — Analysis

In [ ]:
import pandas as pd

# Reload the cleaned dataset
df = pd.read_csv("cleaned_ecommerce_data.csv")

print("Shape:",   df.shape)
print("Columns:", list(df.columns))

In [ ]:
# Re-save with correct encoding so special characters are handled cleanly
df.to_csv("clean_fixed.csv", index=False, encoding="utf-8", quoting=1)

In [ ]:
# Use payment_value as the Revenue column
df["Revenue"] = df["payment_value"]

### 3.1 — Key Business Metrics

In [ ]:
# Calculate headline KPIs
total_orders   = df["order_id"].nunique()
total_revenue  = df["Revenue"].sum()
avg_order_value = df["Revenue"].mean()

print(f"Total Orders      : {total_orders:,}")
print(f"Total Revenue (R$): {total_revenue:,.2f}")
print(f"Avg Order Value   : {avg_order_value:.2f}")

**💡 Business Insight:**
- Average order value gives a baseline for evaluating promotions or upsell strategies.
- Total order count vs. total revenue highlights whether growth comes from volume or higher-value baskets.

### 3.2 — Monthly Revenue Trend

In [ ]:
# Parse dates and extract month information
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])
df["Month_num"] = df["order_purchase_timestamp"].dt.month
df["Month"]     = df["order_purchase_timestamp"].dt.month_name()

# Group revenue by month, preserving calendar order
monthly_sales = (
    df.sort_values("Month_num")
      .groupby("Month", sort=False)["Revenue"]
      .sum()
)

print(monthly_sales.round(2))

**💡 Business Insight:**
- Months with revenue spikes indicate seasonal demand — useful for planning stock and marketing budgets.
- Low-revenue months can be targeted with promotions to smooth out revenue across the year.

### 3.3 — Top 10 Customers by Revenue

In [ ]:
# Rank customers by total spend and show top 10
top_customers = (
    df.groupby("customer_unique_id")["Revenue"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

top10_share = (top_customers.sum() / df["Revenue"].sum()) * 100

print("\n===== TOP 10 CUSTOMERS =====")
print(top_customers.round(2))
print(f"\nTop 10 Revenue Contribution: {top10_share:.2f}%")

**💡 Business Insight:**
- If the top 10 customers account for a large share of revenue, the business is at risk from customer churn.
- These high-value customers are ideal candidates for a loyalty or retention programme.

### 3.4 — Top Product Categories by Revenue

In [ ]:
# Sum revenue per product category and show the top 10
category_sales = (
    df.groupby("product_category_name")["Revenue"]
      .sum()
      .sort_values(ascending=False)
)

print("\n===== TOP 10 CATEGORIES =====")
print(category_sales.head(10).round(2))

**💡 Business Insight:**
- The top categories drive the majority of revenue — these should be prioritised for inventory and promotions.
- Underperforming categories may need pricing review or better visibility on the platform.

### 3.5 — Revenue by Payment Method

In [ ]:
# Analyse which payment types contribute most to revenue
payment_analysis = df.groupby("payment_type")["Revenue"].sum()

print("\n===== PAYMENT METHOD ANALYSIS =====")
print(payment_analysis.round(2))

**💡 Business Insight:**
- Dominant payment methods should be kept smooth and optimised to avoid cart abandonment.
- Low-usage payment types may indicate customer friction — worth investigating.

### 3.6 — Delivery Time Analysis

In [ ]:
# Calculate actual delivery time in days (purchase → delivery)
df["delivery_time"] = (
    pd.to_datetime(df["order_delivered_customer_date"]) -
    pd.to_datetime(df["order_purchase_timestamp"])
).dt.days

avg_delivery = df["delivery_time"].mean()

print("\n===== DELIVERY TIME =====")
print(f"Average Delivery Time: {avg_delivery:.2f} days")

In [ ]:
# Identify late orders: delivered after estimated date
df["delay"] = (
    pd.to_datetime(df["order_delivered_customer_date"]) -
    pd.to_datetime(df["order_estimated_delivery_date"])
).dt.days

late_orders      = df[df["delay"] > 0].shape[0]
late_order_pct   = (late_orders / df["order_id"].nunique()) * 100

print("\n===== DELIVERY DELAYS =====")
print(f"Late Orders : {late_orders:,}")
print(f"Late Order % : {late_order_pct:.2f}%")

**💡 Business Insight:**
- High late-order percentages damage customer satisfaction and increase support requests.
- Average delivery time benchmarks the logistics partner — useful input for SLA negotiations.

### 3.7 — Revenue by Customer State (Top 10)

In [ ]:
# Group revenue by Brazilian state and rank the top 10
state_sales = (
    df.groupby("customer_state")["Revenue"]
      .sum()
      .sort_values(ascending=False)
)

print("\n===== TOP 10 STATES BY REVENUE =====")
print(state_sales.head(10).round(2))

**💡 Business Insight:**
- Revenue concentration in a few states suggests an opportunity to expand marketing in lower-performing regions.
- Top states should receive priority in logistics and delivery partnerships.

## 📈 Section 4 — Visualisation

> The plots below use the same chart types from the original analysis. Only titles, axis labels, and figure size have been tidied up for readability.

In [ ]:
import matplotlib.pyplot as plt

# --- Plot 1: Monthly Revenue Trend ---
plt.figure(figsize=(10, 5))
monthly_sales.plot(kind="line", marker="o", color="steelblue")

plt.title("Monthly Revenue Trend", fontsize=14, fontweight="bold")
plt.xlabel("Month", fontsize=11)
plt.ylabel("Total Revenue (R$)", fontsize=11)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 2: Top 10 Product Categories by Revenue ---
plt.figure(figsize=(12, 5))
category_sales.head(10).plot(kind="bar", color="coral")

plt.title("Top 10 Product Categories by Revenue", fontsize=14, fontweight="bold")
plt.xlabel("Product Category", fontsize=11)
plt.ylabel("Total Revenue (R$)", fontsize=11)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 3: Revenue by Payment Method ---
plt.figure(figsize=(7, 4))
payment_analysis.plot(kind="bar", color="mediumseagreen")

plt.title("Revenue by Payment Method", fontsize=14, fontweight="bold")
plt.xlabel("Payment Type", fontsize=11)
plt.ylabel("Total Revenue (R$)", fontsize=11)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 4: Top 10 Customers by Revenue ---
plt.figure(figsize=(12, 5))
top_customers.plot(kind="bar", color="mediumpurple")

plt.title("Top 10 Customers by Revenue", fontsize=14, fontweight="bold")
plt.xlabel("Customer ID", fontsize=11)
plt.ylabel("Total Revenue (R$)", fontsize=11)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## ✅ Section 5 — Final Summary & Key Findings

| # | Finding |
|---|---------|
| 1 | **Seasonal demand** — Certain months show significantly higher revenue, indicating seasonal buying patterns. |
| 2 | **Category concentration** — A small number of product categories drive the majority of total revenue. |
| 3 | **Customer concentration** — The top 10 customers contribute a notable share of revenue, flagging retention risk. |
| 4 | **Payment preference** — Credit card / instalment plans dominate transactions, while other methods lag behind. |
| 5 | **Delivery delays** — A portion of orders arrive after the estimated date, which is a key customer-experience risk. |
| 6 | **Geographic concentration** — Revenue is heavily skewed toward a few Brazilian states, presenting regional expansion opportunity. |

---
*Dataset: Olist Brazilian E-Commerce — publicly available on Kaggle.*